# The Determinants of Turnout

In [1]:
import stata_setup

stata_setup.config('/Applications/StataNow', 'mp')


  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      StataNow 19.5
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2025 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Unlimited-user 2-core network, expiring 31 Oct 2026
Serial number: 501909305069
  Licensed to: Yichen Luo
               UCL

Notes:
      1. Unicode is supported; see help unicode_advice.
      2. More than 2 billion observations are allowed; see help obs_advice.
      3. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


In [2]:
%%stata

clear all
set more off
set varabbrev off
version 18

* Paths
global PROCESSED_DATA "/Users/yichenluo/Library/CloudStorage/Dropbox/dao-governance/processed_data"
global TABLES "tables"


. 
. clear all

. set more off

. set varabbrev off

. version 18

. 
. * Paths
. global PROCESSED_DATA "/Users/yichenluo/Library/CloudStorage/Dropbox/dao-gove
> rnance/processed_data"

. global TABLES "tables"

. 


In [3]:
%%stata

import delimited using "$PROCESSED_DATA/proposals_panel.csv", clear

* Parse date
gen day = date(date, "YMD")
format day %td
gen month = month(day)
gen quarter = quarter(day)

gen year = year(day)
encode gecko_id, gen(token)
encode space,    gen(dao)


replace concensus = concensus * have_discussion
replace professionalism = professionalism * have_discussion

* Independent variables
local complexity multi_choices weighted quorum
local discussion_char concensus professionalism
local topic protocol_security treasury_expenditure user_incentive_increase tokenomics voting_mechanism_change


* Label variables
label var weighted "\${\it Weighted}_{i,t}\$"
label var quadratic "\${\it Quadratic Voting}_{i,t}\$"
label var ranked_choice "\${\it Ranked Choice}_{i,t}\$"
label var quorum "\${\it Quorum}_{i,t}\$"
label var multi_choices "\${\it Multiple Choices}_{i,t}\$"
label var have_discussion "\${\it Discussion}_{i,t}\$"
label var delegation "\${\it Delegation}_{i,t}\$"
label var professionalism  "\${\it Professionalism}_{i,t}\$"
label var concensus       "\${\it Concensus}_{i,t}\$"
label var non_whale_user      "\${\it User}_{i,t}^{\it Small}\$"
label var whale_user      "\${\it User}_{i,t}^{\it Whale}\$"
label var voter_user      "\${\it User}_{i,t}\$"
label var concensus_diff "\${\it \Delta Concensus}_{i,t}\$"


. 
. import delimited using "$PROCESSED_DATA/proposals_panel.csv", clear
(encoding automatically selected: ISO-8859-1)
(65 vars, 2,830 obs)

. 
. * Parse date
. gen day = date(date, "YMD")

. format day %td

. gen month = month(day)

. gen quarter = quarter(day)

. 
. gen year = year(day)

. encode gecko_id, gen(token)

. encode space,    gen(dao)

. 
. 
. replace concensus = concensus * have_discussion
(219 real changes made)

. replace professionalism = professionalism * have_discussion
(218 real changes made)

. 
. * Independent variables
. local complexity multi_choices weighted quorum

. local discussion_char concensus professionalism

. local topic protocol_security treasury_expenditure user_incentive_increase to
> kenomics voting_mechanism_change

. 
. 
. * Label variables
. label var weighted "\${\it Weighted}_{i,t}\$"

. label var quadratic "\${\it Quadratic Voting}_{i,t}\$"

. label var ranked_choice "\${\it Ranked Choice}_{i,t}\$"

. label var quorum "\${\it Quorum}_{i,t}\$

## Proposal Characteristics

In [4]:
%%stata

******************************************************
* PANEL REGRESSIONS
******************************************************

eststo clear

    * VIF test
    qui reg non_whale_from_delegation_rate `discussion' `complexity' `topic'
    estat vif

    * Run baseline regression
    reghdfe non_whale_from_delegation_rate `complexity' have_discussion , absorb(year token) 
    eststo baseline_t_t
    estadd local fe_token "Y"
    estadd local fe_proposal  "N"
    estadd local fe_time  "Y"

    reghdfe non_whale_from_delegation_rate `complexity' have_discussion , absorb(year) 
    eststo baseline_t
    estadd local fe_token "N"
    estadd local fe_proposal  "N"
    estadd local fe_time  "Y"


    reghdfe non_whale_from_delegation_rate `complexity' have_discussion , absorb(year categories) 
    eststo baseline_p
    estadd local fe_token "N"
    estadd local fe_proposal  "Y"
    estadd local fe_time  "Y"


    * Run regression with discussion characteristics    
    reghdfe non_whale_from_delegation_rate `complexity' concensus_diff, absorb(year token) 
    eststo discussion_t_t
    estadd local fe_token "Y"
    estadd local fe_proposal  "N"
    estadd local fe_time  "Y"

    reghdfe non_whale_from_delegation_rate `complexity' concensus_diff, absorb(year) 
    eststo discussion_t
    estadd local fe_token "N"
    estadd local fe_proposal  "N"
    estadd local fe_time  "Y"

    reghdfe non_whale_from_delegation_rate `complexity' concensus_diff, absorb(year categories) 
    eststo discussion_p
    estadd local fe_token "N"
    estadd local fe_proposal  "Y"
    estadd local fe_time  "Y"


    * Run regression with user characteristics
    reghdfe non_whale_from_delegation_rate `complexity' have_discussion voter_user, absorb(year token) 
    eststo user_t_t
    estadd local fe_token "Y"
    estadd local fe_proposal  "N"
    estadd local fe_time  "Y"

    reghdfe non_whale_from_delegation_rate `complexity' have_discussion voter_user, absorb(year) 
    eststo user_t
    estadd local fe_token "N"
    estadd local fe_proposal  "N"
    estadd local fe_time  "Y"

    reghdfe non_whale_from_delegation_rate `complexity' have_discussion voter_user, absorb(year categories)
    eststo user_p
    estadd local fe_token "N"
    estadd local fe_proposal  "Y"
    estadd local fe_time  "Y"

* Export LaTeX table
esttab                                                           ///
    baseline_t_t baseline_t baseline_p discussion_t_t discussion_t discussion_p user_t_t user_t user_p ///
    using "$TABLES/reg_delegation.tex", replace          ///
    se star(* 0.10 ** 0.05 *** 0.01) label nogaps nocon nonumbers ///
    nonotes booktabs                                             ///
    b(%9.3f) se(%9.2f)                                           ///
    mgroups("\${\it Delegation}^{Small}_{i,t}\$"              ///
            span                                                 ///
            pattern(1 0 0 0 0 0 0 0 0)                           ///
            prefix(\multicolumn{@span}{c}{)                      ///
            suffix(})                                            ///
            erepeat(\cmidrule(lr){@span}) )                      ///
    mtitles("Full" "Full" "Full"                                ///
            "Discussion $\geq$ 2" "Discussion $\geq$ 2" "Discussion $\geq$ 2" ///
            "Have Dapp" "Have Dapp" "Have Dapp")                ///
            substitute("\_" "_")                                ///
    stats(fe_token fe_proposal fe_time N r2_a,                               ///
        fmt(0 0 0 %9.0fc %9.3f)                                    ///
         labels("Token FE" "Category FE" "Year FE" "Observations" "Adjusted R²"))


. 
. ******************************************************
. * PANEL REGRESSIONS
. ******************************************************
. 
. eststo clear

. 
.     * VIF test
.     qui reg non_whale_from_delegation_rate `discussion' `complexity' `topic'

.     estat vif

    Variable |       VIF       1/VIF  
-------------+----------------------
    weighted |      1.71    0.585846
multi_choi~s |      1.46    0.683660
  tokenomics |      1.31    0.760482
      quorum |      1.29    0.776524
user_incen~e |      1.26    0.794557
protocol_s~y |      1.10    0.905396
voting_mec~e |      1.04    0.960474
treasury_e~e |      1.04    0.963081
-------------+----------------------
    Mean VIF |      1.28

. 
.     * Run baseline regression
.     reghdfe non_whale_from_delegation_rate `complexity' have_discussion , abs
> orb(year token) 
(MWFE estimator converged in 6 iterations)

HDFE Linear regression                            Number of obs   =        682
Absorbing 2 HDFE groups         